# SERVERLOGANAMOLY
## Server Log Anomaly Detection with Isolation Forest

### Goal
Parse Apache access logs, aggregate by IP/minute, engineer behavioral features, inject controlled attacks, train Isolation Forest, and evaluate contamination values from 0.01 to 0.05.

In [2]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.log_anomaly import parse_log, engineer_features, train_model, FEATURES, make_report

if not (ROOT / "data/raw/access.log").exists():
    ROOT = ROOT.parent

raw = parse_log(ROOT / "data/raw/access.log")
raw.head()

,ip,timestamp,method,url,status,bytes,window
0,192.0.2.31,2026-09-01 09:00:00+00:00,POST,/api/search,200,13138,2026-09-01 09:00:00+00:00
1,192.0.2.28,2026-09-01 09:00:00+00:00,POST,/api/items,404,3340,2026-09-01 09:00:00+00:00
2,192.0.2.49,2026-09-01 09:00:00+00:00,GET,/contact,200,5557,2026-09-01 09:00:00+00:00
3,192.0.2.54,2026-09-01 09:00:00+00:00,GET,/favicon.ico,200,152,2026-09-01 09:00:00+00:00
4,192.0.2.33,2026-09-01 09:00:00+00:00,GET,/,200,15862,2026-09-01 09:00:00+00:00


## 1. Parse logs

In [3]:
print('Parsed lines:', len(raw))
print('Unique IPs:', raw.ip.nunique())
print(raw.dtypes)

Parsed lines: 52904
Unique IPs: 54
ip                           str
timestamp    datetime64[us, UTC]
method                       str
url                          str
status                     int64
bytes                      int64
window       datetime64[us, UTC]
dtype: object


## 2. Engineer IP + 1-minute features

In [4]:
feat=engineer_features(raw)
feat.head()

,ip,window,requests_per_minute,error_404_ratio,unique_urls,avg_payload_size,post_share
0,192.0.2.10,2026-09-01 09:00:00+00:00,11,0.090909,8,9250.090909,0.000000
1,192.0.2.10,2026-09-01 09:01:00+00:00,13,0.153846,9,8728.230769,0.076923
2,192.0.2.10,2026-09-01 09:02:00+00:00,8,0.250000,8,8470.375000,0.125000
3,192.0.2.10,2026-09-01 09:03:00+00:00,9,0.222222,7,8417.888889,0.111111
4,192.0.2.10,2026-09-01 09:04:00+00:00,9,0.333333,7,10082.222222,0.111111


In [5]:
feat[FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
requests_per_minute,6132.0,8.627528,8.636475,1.000000,6.000000,8.000000,10.0000,600.0
error_404_ratio,6132.0,0.181218,0.146364,0.000000,0.083333,0.166667,0.2500,1.0
unique_urls,6132.0,6.701239,2.044867,1.000000,5.000000,7.000000,8.0000,13.0
avg_payload_size,6132.0,9058.600303,1943.271299,270.104167,7830.600000,9065.600000,10317.0375,17375.0
post_share,6132.0,0.118501,0.119555,0.000000,0.000000,0.111111,0.2000,1.0


## 3. Isolation Forest + contamination comparison

In [6]:
rows=[]
attack_ips={'198.51.100.10','198.51.100.11','198.51.100.12'}
for c in [0.01,0.02,0.03,0.04,0.05]:
    _,_,res=train_model(feat,c)
    injected=res.ip.isin(attack_ips)
    tp=int((res.anomaly & injected).sum())
    attacks=int(injected.sum())
    fp=int((res.anomaly & ~injected).sum())
    normal=int((~injected).sum())
    rows.append({'contamination':c,'attack_windows':attacks,'attack_windows_flagged':tp,'detection_rate':tp/attacks,'false_positives':fp,'false_positive_rate':fp/normal})
eval_df=pd.DataFrame(rows)
eval_df

,contamination,attack_windows,attack_windows_flagged,detection_rate,false_positives,false_positive_rate
0,0.01,5,5,1.0,57,0.009303
1,0.02,5,5,1.0,118,0.019259
2,0.03,5,5,1.0,179,0.029215
3,0.04,5,5,1.0,241,0.039334
4,0.05,5,5,1.0,302,0.049290


### Contamination choice
The default is 0.03 for this controlled fixture. It detects all injected attack windows while keeping the false-positive rate near the configured anomaly budget. On real traffic, tune this value using representative validation data and operational alert capacity.

## 4. Train final model and create Threat Intelligence Report

In [7]:
model,scaler,result=train_model(feat,0.03)
flagged=result[result.anomaly]
print('Flagged windows:', len(flagged))
flagged.sort_values('anomaly_score').head(20)

Flagged windows: 184


,ip,window,requests_per_minute,error_404_ratio,unique_urls,avg_payload_size,post_share,anomaly,anomaly_score
5959,192.0.2.59,2026-09-01 10:12:00+00:00,1,1.000000,1,12013.000000,1.000000,True,-0.174487
6127,198.51.100.10,2026-09-01 09:30:00+00:00,240,1.000000,3,270.104167,0.000000,True,-0.153943
3484,192.0.2.38,2026-09-01 11:00:00+00:00,1,1.000000,1,16838.000000,0.000000,True,-0.151885
1740,192.0.2.24,2026-09-01 09:59:00+00:00,2,1.000000,2,11665.500000,0.500000,True,-0.146591
843,192.0.2.17,2026-09-01 09:02:00+00:00,1,0.000000,1,3232.000000,1.000000,True,-0.142360
6129,198.51.100.11,2026-09-01 09:41:00+00:00,30,1.000000,8,471.866667,0.000000,True,-0.141805
6128,198.51.100.11,2026-09-01 09:40:00+00:00,30,1.000000,8,458.333333,0.000000,True,-0.141805
1832,192.0.2.25,2026-09-01 09:31:00+00:00,1,1.000000,1,13509.000000,0.000000,True,-0.137171
2466,192.0.2.30,2026-09-01 10:04:00+00:00,2,0.500000,2,3450.000000,0.500000,True,-0.127901
2642,192.0.2.31,2026-09-01 11:00:00+00:00,1,1.000000,1,6895.000000,0.000000,True,-0.124896


In [8]:
out=ROOT/'reports/notebook_threat_report.md'
make_report(result,raw,out)
print(out)

c:\Users\PRAGATI\Downloads\ServerLogAnomoly\ServerLogAnomoly\reports\notebook_threat_report.md


## 5. Inference workflow
Use the packaged CLI:

`python infer.py path/to/new_access.log --contamination 0.03 --report reports/inference_report.md`